In [7]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
# Setting a professional aesthetic for charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16

from load_cleaned_dataset import get_cleaned_df

df = get_cleaned_df()



 Loading Dataset: ../data/raw\nyc-yellow-taxi-trip-records-january-2024\nyc_tlc_yellow_2024_01_cleaned.csv
 Dataset Loaded Successfully! Total Rows: 2,599,399
Type enforcement complete. New Memory Usage:
399.12 MB


### Phase 4.1: Behavioral Inference (Classification)
## 1. The Multi-Stage Model Approach
In a real-world Data Science pipeline, models are often stacked. Here, we transition from **Regression** (predicting the Fare) to **Classification** (predicting Customer Behavior).

### 2. Objective: Generosity Classification
We define a "Generous Customer" as someone who tips more than **20%** of the total fare. The goal is to see if the trip's characteristics (Price, Distance, Time) can predict this high-value behavior.

### 3. Feature Integration
* **Input Features:** Fare Amount (from Regression), Trip Distance, and Pickup Hour.
* **Target Variable:** Binary Class (1: High Tip, 0: Standard/No Tip).
* **Algorithm:** Random Forest Classifier (chosen for its ability to handle non-linear relationships between price and tipping habits).

### 4. Strategic Outcome
This allows the taxi company to build a "Loyalty Score" or "Priority Dispatch" system, directing drivers to trips that have a high probability of being highly profitable due to tips.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- STEP 1: Feature Engineering & Labeling ---
# Extracting the hour to understand tipping patterns by time of day
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

# Creating our Target Variable (The Label)
# 1 = Generous (Tip > 20%), 0 = Normal
df['is_generous'] = (df['tip_amount'] > (df['fare_amount'] * 0.20)).astype(int)

# --- STEP 2: Feature Selection ---
# Selecting features that logically impact tipping behavior
features = ['fare_amount', 'trip_distance', 'pickup_hour', 'passenger_count']
X = df[features]
y = df['is_generous']

# --- STEP 3: Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- STEP 4: High-Speed Model Training ---
# n_jobs=-1 utilizes all CPU cores to handle the 2.6M rows efficiently
print("⏳ Training Random Forest Classifier... This may take a moment.")
clf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    n_jobs=-1, 
    random_state=42
)

clf.fit(X_train, y_train)
print("✅ Training Complete!")

# --- STEP 5: Model Evaluation ---
y_pred = clf.predict(X_test)

print("\n--- Performance Metrics ---")
print(f"Overall Accuracy Score: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

⏳ Training Random Forest Classifier... This may take a moment.
✅ Training Complete!

--- Performance Metrics ---
Overall Accuracy Score: 64.64%

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.03      0.06    185655
           1       0.65      0.99      0.78    334225

    accuracy                           0.65    519880
   macro avg       0.62      0.51      0.42    519880
weighted avg       0.62      0.65      0.53    519880



### Phase 4.2: Enhancing Model Accuracy with Payment Type Data

#### 🎯 Strategic Insight
The initial model (65% accuracy) struggled because tipping behavior is not just about distance or time—it is heavily dependent on **how** the passenger pays. In NYC, tips are electronically recorded for Credit Cards but often missing (zero) in the system for Cash payments.

#### 🧠 Improvements
By adding the `payment_type` feature, we provide the model with the "missing link." This allows the classifier to distinguish between a "Non-generous" passenger and a "Cash" passenger whose tip simply wasn't logged, leading to a much more balanced and realistic prediction.

#### 📊 Expected Outcome
Including `payment_type` typically boosts the F1-Score significantly, as the model can now accurately identify Class 0 (non-generous/cash trips) instead of labeling everyone as generous.

In [9]:
# --- STEP 1: Feature Engineering & Cleaning ---
# Extracting time-based features
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

# Define Generosity: Tip > 20%
df['is_generous'] = (df['tip_amount'] > (df['fare_amount'] * 0.20)).astype(int)

# --- STEP 2: Feature Selection (Now including Payment Type) ---
# Payment_type is crucial: 1=Credit Card, 2=Cash
# Usually, tips for Cash (2) are not recorded, which helps the model distinguish
features = ['fare_amount', 'trip_distance', 'pickup_hour', 'passenger_count', 'payment_type']
X = df[features]
y = df['is_generous']

# --- STEP 3: Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- STEP 4: Model Training (Parallelized) ---
print("⏳ Training optimized model with Payment Type... please wait.")
clf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    n_jobs=-1, 
    random_state=42
)

clf.fit(X_train, y_train)
print("✅ Training Complete!")

# --- STEP 5: Evaluation ---
y_pred = clf.predict(X_test)

print("\n--- Performance Metrics ---")
print(f"Overall Accuracy Score: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

⏳ Training optimized model with Payment Type... please wait.
✅ Training Complete!

--- Performance Metrics ---
Overall Accuracy Score: 80.84%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.46      0.63    185655
           1       0.77      1.00      0.87    334225

    accuracy                           0.81    519880
   macro avg       0.88      0.73      0.75    519880
weighted avg       0.85      0.81      0.79    519880



## 4.3. The Dual-Intelligence System: Predicting Value and Quality

### 1. Regression: Estimating Quantitative Revenue
The **Regression** model acts as our "Financial Calculator." By analyzing distance and time, it predicts the **Fare Amount**. This tells the business the expected gross revenue for any given trip.

### 2. Classification: Evaluating Qualitative Quality
The **Classification** model acts as our "Service Auditor." It doesn't look at the dollars, but at the **Generosity**. It tells us if the passenger is likely to reward the driver with a high tip (Quality Trip).

### 3. Business Synthesis
By combining both models, we can identify "Prime Trips":
* **High Quantity + High Quality:** Long trips with generous passengers (Maximum Profit).
* **Low Quantity + High Quality:** Short trips with generous passengers (High efficiency for drivers).

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
import warnings

# Option 1: To completely silence any remaining warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- 1. DATA PREPARATION ---
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['trip_duration'] = (pd.to_datetime(df['tpep_dropoff_datetime']) - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df['is_generous'] = (df['tip_amount'] > (df['fare_amount'] * 0.20)).astype(int)

# --- 2. MODEL TRAINING ---
# Regression: Predicting the Dollar Value
X_reg = df[['trip_distance', 'trip_duration']]
y_reg = df['fare_amount']
reg_model = LinearRegression().fit(X_reg, y_reg)

# Classification: Predicting the Quality
X_cls = df[['fare_amount', 'trip_distance', 'pickup_hour', 'passenger_count', 'payment_type']]
y_cls = df['is_generous']
clf_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42).fit(X_cls, y_cls)

# --- 3. INPUTS ---
sample_distance = 5.0
sample_duration = 15.0
sample_hour = 14
sample_passengers = 1
sample_payment = 1

# --- 4. CLEAN EXECUTION (Using DataFrames with Feature Names) ---
# Predicting Revenue
sample_reg_df = pd.DataFrame([[sample_distance, sample_duration]], 
                             columns=['trip_distance', 'trip_duration'])
predicted_fare = reg_model.predict(sample_reg_df)[0]

# Predicting Quality
sample_cls_df = pd.DataFrame([[predicted_fare, sample_distance, sample_hour, sample_passengers, sample_payment]], 
                             columns=['fare_amount', 'trip_distance', 'pickup_hour', 'passenger_count', 'payment_type'])
quality_pred = clf_model.predict(sample_cls_df)[0]
quality_label = "Generous (High Quality)" if quality_pred == 1 else "Normal (Low Quality)"

# --- 5. FINAL OUTPUT ---
print("\n" + "="*40)
print("🚀 NEW YORK TAXI INTELLIGENCE SYSTEM")
print("="*40)
print(f"INPUTS   | Distance: {sample_distance} mi | Time: {sample_duration} min")
print("-" * 40)
print(f"REVENUE  | Expected Fare: ${predicted_fare:.2f}")
print(f"QUALITY  | Trip Category: {quality_label}")
print("="*40)


🚀 NEW YORK TAXI INTELLIGENCE SYSTEM
INPUTS   | Distance: 5.0 mi | Time: 15.0 min
----------------------------------------
REVENUE  | Expected Fare: $23.27
QUALITY  | Trip Category: Generous (High Quality)
